In [ ]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support
)


In [ ]:
# KONFIGURASI

CSV_PATH = "../dataset/data/tomato_RGB_HSV_ratio_dataset.csv"
LABEL_COL = "Label"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

CLASS_LABELS = ["unripe", "semi-ripe", "fully-ripe"]

TEST_SIZE = 0.2
RANDOM_STATE = 42
KNN_K = 5


In [ ]:
# LOAD DATASET

def load_dataset(csv_path, feature_list):
    df = pd.read_csv(csv_path)

    X = df[feature_list]
    y = df[LABEL_COL]

    return train_test_split(
        X, y,
        test_size=TEST_SIZE,
        shuffle=True,
        random_state=RANDOM_STATE
    )


In [ ]:
# TRAIN & EVALUATE MODEL

def train_and_evaluate(csv_path, feature_list):
    X_train, X_test, y_train, y_test = load_dataset(
        csv_path,
        feature_list
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = KNeighborsClassifier(n_neighbors=KNN_K)
    model.fit(X_train_scaled, y_train)

    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    return {
        "model": model,
        "scaler": scaler,

        "X_train": X_train,
        "y_train": y_train,
        "y_train_pred": y_train_pred,

        "X_test": X_test,
        "y_test": y_test,
        "y_test_pred": y_test_pred,

        "train_acc": accuracy_score(y_train, y_train_pred) * 100,
        "test_acc": accuracy_score(y_test, y_test_pred) * 100
    }


In [ ]:
# SKENARIO FITUR

feature_sets = {
    "RGB": ["R", "G", "B"],
    "HSV": ["H", "S", "V"],
    "Rasio (R/G, R/B)": ["R/G", "R/B"],
    "RGB + Rasio": ["R", "G", "B", "R/G", "R/B"],
    "HSV + Rasio": ["H", "S", "V", "R/G", "R/B"],
    "RGB + HSV + Rasio": ["R", "G", "B", "H", "S", "V", "R/G", "R/B"]
}


In [ ]:
# SELEKSI MODEL TERBAIK

results = []

best_test_acc = 0
best_result = None
best_feature_name = None

for name, features in feature_sets.items():
    r = train_and_evaluate(CSV_PATH, features)

    results.append({
        "Kombinasi Fitur": name,
        "Akurasi Train (%)": round(r["train_acc"], 2),
        "Akurasi Test (%)": round(r["test_acc"], 2),
        "Misclassification Train (%)": round(100 - r["train_acc"], 2),
        "Misclassification Test (%)": round(100 - r["test_acc"], 2)
    })

    if r["test_acc"] > best_test_acc:
        best_test_acc = r["test_acc"]
        best_result = r
        best_feature_name = name

comparison_df = pd.DataFrame(results)

print("\nTABEL PERBANDINGAN KOMBINASI FITUR KNN\n")
display(comparison_df)



TABEL PERBANDINGAN KOMBINASI FITUR KNN



,Kombinasi Fitur,Akurasi Train (%),Akurasi Test (%),Misclassification Train (%),Misclassification Test (%)
0,RGB,89.05,87.72,10.95,12.28
1,HSV,89.18,87.47,10.82,12.53
2,"Rasio (R/G, R/B)",88.99,85.42,11.01,14.58
3,RGB + Rasio,90.27,88.75,9.73,11.25
4,HSV + Rasio,89.88,88.49,10.12,11.51
5,RGB + HSV + Rasio,90.78,89.26,9.22,10.74


In [ ]:
# CELL 7: SIMPAN MODEL TERBAIK

def save_best_model(model, scaler, feature_name):
    safe_name = (
        feature_name.replace(" ", "_")
                    .replace("/", "_")
                    .replace("(", "")
                    .replace(")", "")
                    .replace(",", "")
    )

    path = os.path.join(
        MODEL_DIR,
        f"knn_model.pkl"
    )

    with open(path, "wb") as f:
        pickle.dump(
            {
                "model": model,
                "scaler": scaler,
                "features": feature_name
            },
            f
        )

    return path


print("MODEL TERBAIK")
print("Kombinasi Fitur :", best_feature_name)
print("Akurasi Test    :", round(best_test_acc, 2), "%")

model_path = save_best_model(
    best_result["model"],
    best_result["scaler"],
    best_feature_name
)

print("Model disimpan di:", model_path)


MODEL TERBAIK
Kombinasi Fitur : RGB + HSV + Rasio
Akurasi Test    : 89.26 %
Model disimpan di: models\knn_model.pkl


In [ ]:
# FUNGSI EVALUASI DETAIL

def evaluation_table(y_true, y_pred, title):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=CLASS_LABELS
    )

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        zero_division=0
    )

    rows = []
    for i, cls in enumerate(CLASS_LABELS):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP

        rows.append([
            cls,
            int(TP),
            int(FP),
            int(FN),
            round(prec[i], 2),
            round(rec[i], 2),
            round(f1[i], 2)
        ])

    df = pd.DataFrame(
        rows,
        columns=[
            "Kelas",
            "True Positive",
            "False Positive",
            "False Negative",
            "Precision",
            "Recall",
            "F1-score"
        ]
    )

    print(f"\nHASIL KLASIFIKASI DATA {title}\n")
    display(df)


In [ ]:
# HASIL EVALUASI MODEL TERBAIK

# DATA LATIH
evaluation_table(
    best_result["y_train"],
    best_result["y_train_pred"],
    "LATIH"
)

# DATA UJI
evaluation_table(
    best_result["y_test"],
    best_result["y_test_pred"],
    "UJI"
)



HASIL KLASIFIKASI DATA LATIH



,Kelas,True Positive,False Positive,False Negative,Precision,Recall,F1-score
0,unripe,1003,29,24,0.97,0.98,0.97
1,semi-ripe,191,66,72,0.74,0.73,0.73
2,fully-ripe,224,49,48,0.82,0.82,0.82



HASIL KLASIFIKASI DATA UJI



,Kelas,True Positive,False Positive,False Negative,Precision,Recall,F1-score
0,unripe,270,11,4,0.96,0.99,0.97
1,semi-ripe,33,17,24,0.66,0.58,0.62
2,fully-ripe,46,14,14,0.77,0.77,0.77
